# 분자동역학 실습

**Molecular Dynamics · MD**

원자에 작용하는 힘으로 운동 방정식을 시간 적분해 구조와 수송 성질을 살피는 시뮬레이션.

소재 분야에서 이해하기: 온도별 원자 확산 거동을 시간에 따라 추적한다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [ASE 원자 시뮬레이션 문서](https://wiki.fysik.dtu.dk/ase/)

## 1. 속도 Verlet으로 2차원 MD 돌리기

레너드-존스 상호작용 원자 64개를 실제로 시간 적분합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

n_side, spacing = 8, 1.15
box = n_side * spacing
positions = np.array([[i * spacing, j * spacing] for i in range(n_side) for j in range(n_side)])
positions += rng.normal(0, 0.02, positions.shape)
velocities = rng.normal(0, 0.5, positions.shape)
velocities -= velocities.mean(0)
print('원자 %d개, 상자 %.2f x %.2f (주기 경계)' % (len(positions), box, box))

def forces_and_energy(positions, box, cutoff=2.5):
    delta = positions[:, None, :] - positions[None, :, :]
    delta -= box * np.round(delta / box)                       # 최소 이미지 규약
    distance2 = (delta ** 2).sum(-1)
    np.fill_diagonal(distance2, np.inf)
    mask = distance2 < cutoff ** 2
    inverse6 = np.where(mask, (1.0 / distance2) ** 3, 0.0)
    energy = 4 * (inverse6 ** 2 - inverse6)[mask].sum() / 2
    magnitude = np.where(mask, 24 * (2 * inverse6 ** 2 - inverse6) / distance2, 0.0)
    force = (magnitude[:, :, None] * delta).sum(1)
    return force, energy

In [ ]:
dt, steps = 0.004, 1500
force, potential = forces_and_energy(positions, box)
history = []
for step in range(steps):
    velocities += 0.5 * dt * force
    positions = (positions + dt * velocities) % box
    force, potential = forces_and_energy(positions, box)
    velocities += 0.5 * dt * force
    kinetic = 0.5 * (velocities ** 2).sum()
    history.append((kinetic, potential, kinetic + potential))

history = np.array(history)
plt.plot(history[:, 0], label='kinetic')
plt.plot(history[:, 1], label='potential')
plt.plot(history[:, 2], label='total')
plt.xlabel('step'); plt.ylabel('energy (LJ units)'); plt.legend(); plt.show()
drift = abs(history[-1, 2] - history[0, 2]) / abs(history[0, 2])
print('총 에너지 상대 변화 %.4f (작아야 적분이 안정적입니다)' % drift)
print('평균 온도(2D, k_B=1) %.3f' % (history[:, 0].mean() / len(positions)))

## 2. 시간 간격을 키우면 무너집니다

In [ ]:
for dt_try in (0.002, 0.008, 0.02):
    p, v = positions.copy(), velocities.copy()
    f, _ = forces_and_energy(p, box)
    start = None
    for step in range(300):
        v += 0.5 * dt_try * f
        p = (p + dt_try * v) % box
        f, potential = forces_and_energy(p, box)
        v += 0.5 * dt_try * f
        total = 0.5 * (v ** 2).sum() + potential
        if start is None:
            start = total
    print('dt=%.3f -> 총 에너지 상대 변화 %.4f' % (dt_try, abs(total - start) / abs(start)))
print('\n적분 간격은 가장 빠른 진동 주기보다 충분히 작아야 합니다. MLIP 를 쓸 때도 같은 제약이 있습니다.')

## 3. 동경분포함수(RDF)

In [ ]:
delta = positions[:, None, :] - positions[None, :, :]
delta -= box * np.round(delta / box)
distance = np.sqrt((delta ** 2).sum(-1))
distance = distance[np.triu_indices(len(positions), 1)]
counts, edges = np.histogram(distance, bins=60, range=(0.5, box / 2))
centres = 0.5 * (edges[1:] + edges[:-1])
shell = 2 * np.pi * centres * (edges[1] - edges[0])
density = len(positions) / box ** 2
plt.plot(centres, counts / (shell * density * len(positions) / 2))
plt.xlabel('r (LJ units)'); plt.ylabel('g(r)'); plt.show()
print('첫 봉우리 위치 %.2f -> 최근접 이웃 거리입니다.' % centres[np.argmax(counts)])

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#molecular-dynamics)을 여세요.